In [ ]:
import geopandas as gpd
import numpy as np
from tqdm import tqdm

# Load the shapefile grid
grid = gpd.read_file("/home/stagiaire/Téléchargements/TE3/grille_finale.shp")
print(f"Grid loaded: {len(grid)} cells")

# Load OSM shapefiles for nodes and edges
osm_nodes = gpd.read_file("/home/stagiaire/Téléchargements/TE3/osm_nodes.shp")
osm_edges = gpd.read_file("/home/stagiaire/Téléchargements/TE3/osm_edges.shp")

print(f"OSM nodes loaded: {len(osm_nodes)}")
print(f"OSM edges loaded: {len(osm_edges)}")

# Convert CRS of OSM data to match the grid
osm_nodes = osm_nodes.to_crs(grid.crs)
osm_edges = osm_edges.to_crs(grid.crs)

# Add columns for statistics in the grid
grid['nodes'] = 0
grid['roads'] = 0.0
grid['mean_connections'] = 0.0
grid['min_connections'] = 0.0
grid['max_connections'] = 0.0

# Compute statistics for each grid cell
for idx, row in tqdm(grid.iterrows(), total=len(grid)):
    geometry = row['geometry']

    # Filter nodes inside the cell
    nodes_in_cell = osm_nodes[osm_nodes.intersects(geometry)]
    grid.at[idx, 'nodes'] = len(nodes_in_cell)

    # Filter roads inside the cell
    roads_in_cell = osm_edges[osm_edges.intersects(geometry)]

    # Compute total road length inside the cell
    total_length = sum(route.geometry.intersection(geometry).length for _, route in roads_in_cell.iterrows())

    grid.at[idx, 'roads'] = total_length

    # Compute connection statistics
    if len(nodes_in_cell) > 0:
        connections = nodes_in_cell['street_cou'].values
        grid.at[idx, 'mean_connections'] = connections.mean()
        grid.at[idx, 'min_connections'] = connections.min()
        grid.at[idx, 'max_connections'] = connections.max()

# Final save after processing all cells
grid.to_file("grid.shp")
print("Grid saved with road network statistics!")
